# RQ2 — Repeat length and sequence patterning

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.data import independent_units
from src.plots.figures import plot_cluster_bins, plot_ranked_clusters, plot_metric_points

PERIODIC_SWEEP = "51a_repeat_number"
SCRAMBLED_SWEEP = "51b_repeat_number_rand"
DENSITY = 0.01
STIFFNESS = 2.5
E_VALUES = (2, 4, 8)

periodic = filter_rows(
    load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)
scrambled = filter_rows(
    load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)
periodic_reps = filter_rows(
    load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "replicates"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)
scrambled_reps = filter_rows(
    load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "replicates"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)
for data, label in ((periodic, "repeat"), (scrambled, "scrambled"),
                    (periodic_reps, "repeat"), (scrambled_reps, "scrambled")):
    data["sequence_class"] = label
summaries = pd.concat((periodic, scrambled), ignore_index=True)
independent = independent_units(
    pd.concat((periodic_reps, scrambled_reps), ignore_index=True)
)

## Cluster-size distribution by repeat length

In [ ]:
fig, axes = panels_by(
    periodic, by="E_att",
    plot_fn=partial(
        plot_cluster_bins, scheme="particle_percentage",
        x="repeat_count", style=STYLE,
    ),
    style=STYLE, ncols=3, legend="last",
)
plt.show()

## Periodic and scrambled sequences — ranked clusters

In [ ]:
fig, axes = plt.subplots(
    2, len(E_VALUES), figsize=(13, 6.7),
    layout="constrained", squeeze=False, sharey=True,
)
for col, strength in enumerate(E_VALUES):
    for row, (sequence, data) in enumerate(
        (("repeat", periodic), ("scrambled", scrambled))
    ):
        ax = axes[row, col]
        subset = filter_rows(data, E_att=strength)
        plot_ranked_clusters(
            subset, ax, x="repeat_count", style=STYLE,
            annotate_molecules=True,
        )
        if row == 0:
            ax.set_title(rf"$E_{{\mathrm{{att}}}}={strength:g}$")
            ax.set_xlabel("")
        if col > 0:
            ax.set_ylabel("")
    axes[1, col].set_xlabel("CAG repeats")

axes[0, 0].annotate("Periodic", xy=(-.28, .5),
                    xycoords="axes fraction", rotation=90,
                    ha="center", va="center")
axes[1, 0].annotate("Scrambled", xy=(-.28, .5),
                    xycoords="axes fraction", rotation=90,
                    ha="center", va="center")
handles, labels = axes[0, -1].get_legend_handles_labels()
for ax in axes.flat:
    legend = ax.get_legend()
    if legend:
        legend.remove()
axes[0, -1].legend(handles, labels, loc="upper left",
                   bbox_to_anchor=(1.02, 1), frameon=False)
plt.show()

## Periodic and scrambled sequences — mean edge weight

In [ ]:
palette = {"repeat": "#D95C69", "scrambled": "#468EBF"}
fig, axes = panels_by(
    independent, by="E_att",
    plot_fn=partial(
        plot_metric_points, metric="mean_edge_weight",
        x="repeat_count", hue="sequence_class",
        palette=palette, style=STYLE,
    ),
    style=STYLE, ncols=3, legend="last",
)
plt.show()

## Permutation tests

In [ ]:
RUN_PERMUTATION_TESTS = False
if RUN_PERMUTATION_TESTS:
    from scipy.stats import permutation_test
    from statsmodels.stats.multitest import multipletests

    comparisons = []
    for (length, e_att), group in independent.groupby(["repeat_count", "E_att"]):
        a = group.loc[group.sequence_class == "repeat", "mean_edge_weight"].dropna().to_numpy()
        b = group.loc[group.sequence_class == "scrambled", "mean_edge_weight"].dropna().to_numpy()
        if len(a) >= 2 and len(b) >= 2:
            test = permutation_test(
                (a, b), statistic=lambda x, y: x.mean() - y.mean(),
                alternative="two-sided", permutation_type="independent",
                n_resamples=10_000, random_state=42,
            )
            comparisons.append({"repeat_count": length, "E_att": e_att, "p": test.pvalue})
    tests = pd.DataFrame(comparisons)
    if not tests.empty:
        tests["p_holm"] = multipletests(tests["p"], method="holm")[1]
        display(tests.sort_values(["repeat_count", "E_att"]))

## Final-frame XY projections

In [ ]:
SHOW_PROJECTIONS = False
if SHOW_PROJECTIONS:
    from src.plots.projections import plot_projection_sweep

    GSD_ROOT = ROOT.parent / "validation" / "bulk" / "output_dir"
    fig, axes = plot_projection_sweep(
        GSD_ROOT / PERIODIC_SWEEP, row_by="repeat_count",
        density=DENSITY, attractions=E_VALUES, style=STYLE,
    )
    plt.show()